# 数据驱动决策（ACCA 先修课）· Decision Making with Data
## 第 5 周：时间序列分析

**授课教师**：王奇 副教授（四川大学商学院）  
**研究方向**：人工智能与公司金融  
**邮箱**：qiwangphd@scu.edu.cn

---

### 本周学习目标
1. 理解时间序列四个成分：趋势 T、季节 S、周期 C、随机 R
2. 掌握移动平均法（含中心化）提取趋势
3. 掌握季节指数的计算与预测应用
4. 用 Python 实现完整的时间序列分解与预测流程

In [ ]:
import numpy as np                                # 数值计算
import pandas as pd                               # 时间序列处理
import matplotlib.pyplot as plt                   # 绘图
from sklearn.linear_model import LinearRegression # 趋势外推用线性回归

plt.rcParams['font.sans-serif'] = ['Songti SC', 'SimHei', 'PingFang SC']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False        # 负号显示

## 1. 案例数据：服装店季度销售额

观察规律：**趋势上升**（Q4 峰值 120→135→150）+ **Q4 最高、Q2 最低的季节性** + 波动幅度随趋势增大 → 用**乘法模型** $Y = T \times S \times C \times R$

In [ ]:
data = [80, 60, 70, 120,                          # 2021 年 Q1-Q4 销售额（万元）
        90, 65, 75, 135,                          # 2022 年 Q1-Q4
        100, 72, 85, 150]                         # 2023 年 Q1-Q4

quarters = [f'{y}Q{q}' for y in range(2021, 2024) for q in range(1, 5)]   # 生成 12 个季度标签

df = pd.DataFrame({'Quarter': quarters, 'Y': data})   # 构建 DataFrame
df['Q_Num'] = [1, 2, 3, 4] * 3                    # 标注每个观测的季度编号（用于分组求季节指数）
df                                              # 显示数据表

## 2. 移动平均提取趋势

**黄金法则**：移动平均期数 = 季节周期长度（季度数据用 4 期）。  
偶数周期需**中心化**（两个连续 4 期 MA 再平均），结果才能对齐到具体季度。  
**原理**：完整周期内季节指数之和为 0，平均后抵消，只剩趋势。

In [ ]:
# 手动验证 2021Q3 的中心化移动平均（课堂例题）
A1 = (80 + 60 + 70 + 120) / 4                     # 第一个 4 期 MA（落在 Q2-Q3 之间）
A2 = (60 + 70 + 120 + 90) / 4                     # 第二个 4 期 MA（落在 Q3-Q4 之间）
CMA_2021Q3 = (A1 + A2) / 2                        # 两者平均 → 恰好对齐 2021Q3（趋势值）

print(f'A1 = {A1}, A2 = {A2}')                     # 打印两个 4 期 MA
print(f'2021Q3 趋势值 CMA = {CMA_2021Q3}')          # 应为 83.75
print(f'实际值 70 − 趋势 83.75 = {70 - 83.75}（负数 = 淡季低于趋势）')   # 季节变动的直观理解

In [ ]:
# ===== Pandas 一行流：rolling 实现中心化移动平均 =====
df['MA4'] = df['Y'].rolling(window=4).mean()      # 4 期移动平均（未中心化）
df['Trend'] = df['MA4'].rolling(window=2).mean()  # 再做 2 期平均 → 中心化移动平均 = 趋势 T
df.round(2)                                      # 保留 2 位小数显示

## 3. 提取季节指数（乘法模型）

季节指数 $S = Y \div T$：大于 1 为旺季，小于 1 为淡季。同一季度跨年取平均得「典型指数」。

In [ ]:
df['Seasonal'] = df['Y'] / df['Trend']            # 计算每期的季节比率 S = Y/T

valid = df.dropna(subset=['Trend'])               # 去掉首尾没有趋势值的行（dropna）
seasonal_index = valid.groupby('Q_Num')['Seasonal'].mean()   # 按季度分组求平均季节指数

print('各季度平均季节指数：')                       # 标题
print(seasonal_index.round(3))                    # Q1≈0.976, Q2≈0.678, Q3≈0.793, Q4≈1.371
print('\n解读：Q4 是旺季（1.371），Q2 是淡季（0.678）')   # 业务解读

## 4. 趋势外推与预测

预测公式（乘法模型）：$\hat{Y} = T \times \bar{S}_{\text{对应季度}}$

In [ ]:
# ===== 用趋势值对时间序号做线性回归 =====
t = np.arange(1, len(valid) + 1).reshape(-1, 1)   # 时间序号 t = 1~10（sklearn 需二维）
trend_model = LinearRegression()                  # 创建回归模型
trend_model.fit(t, valid['Trend'].values)         # 拟合趋势线 T = a + b·t

print(f'趋势方程: T = {trend_model.intercept_:.2f} + {trend_model.coef_[0]:.2f} × t')   # 打印趋势方程

# ===== 预测 2024 年 4 个季度 =====
future_t = np.array([[10], [11], [12], [13]])     # 未来 4 期的时间序号
future_trend = trend_model.predict(future_t)      # 外推趋势值 T
future_q = [1, 2, 3, 4]                           # 对应的季度编号
forecast = future_trend * [seasonal_index[q] for q in future_q]   # 预测 = 趋势 × 季节指数

for q, tr, fc in zip(future_q, future_trend, forecast):   # 逐行打印预测结果
    print(f'2024Q{q}: 趋势={tr:.1f}, 季节指数={seasonal_index[q]:.3f}, 预测={fc:.1f} 万元')   # 明细
print(f'\n2024 全年预测销售额 ≈ {forecast.sum():.0f} 万元')   # 全年合计（约 488）

In [ ]:
# ===== 完整可视化：实际值 + 趋势 + 预测 =====
plt.figure(figsize=(12, 6))                        # 大画布
plt.plot(df['Y'], 'o-', color='#8B6F4E', label='实际值')            # 实际值（圆点实线）
plt.plot(df['Trend'], 's--', color='#C49A6C', label='趋势 (CMA)')   # 趋势（方块虚线）
plt.plot(range(9, 13), forecast, '^--', color='#E74C3C', label='2024 预测')   # 预测（三角虚线）
plt.xticks(range(12), quarters, rotation=45)      # x 轴刻度用季度标签并旋转 45 度
plt.xlabel('季度')                                 # x 轴标签
plt.ylabel('销售额（万元）')                        # y 轴标签
plt.title('时间序列分解与预测：实际值 → 趋势 → 季节调整预测')   # 标题
plt.legend()                                      # 图例
plt.grid(True, alpha=0.3)                         # 网格
plt.tight_layout()                                # 防止标签被裁剪
plt.show()                                        # 显示

## 5. 加法 vs 乘法模型速查

| 模型 | 公式 | 适用场景 | 季节成分 |
|---|---|---|---|
| 加法 | $Y = T + S + C + R$ | 波动**绝对幅度**固定（气温、库存量）| $S = Y - T$ |
| 乘法 | $Y = T \times S \times C \times R$ | 波动**相对比例**固定（销售额、利润）| $S = Y \div T$ |

## 6. 本周小结

**完整流程**：原始数据 → 4期移动平均 → 中心化（对齐季度）→ 季节指数（按季度平均）→ 趋势外推（线性回归）→ 预测（趋势 × 季节指数）

- 移动平均期数 = 季节周期长度（黄金法则）
- 偶数周期必须中心化，否则趋势值落在两期之间
- 数学原理：完整周期内 S 之和为 0（加法）→ 平均后季节抵消，只剩趋势